코드 stable-diffusion-basic

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

# 사전학습 모델 로드 (첫 실행 시 약 4GB 다운로드)
pipe = StableDiffusionPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=torch.float16
).to('cuda')

prompt = '한복을 입은 한국 여성, 가을 단풍 배경, 사진 사실주의 스타일'
negative_prompt = '저해상도, 흐릿함, 왜곡된 얼굴'

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=50,
    guidance_scale=7.5,
    height=512, width=512
).images[0]

image.save('output.png')

코드 deepfake-detector

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

class DeepfakeDetector(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        # EfficientNet-B0: 작고 효율적, GPU에서 빠른 학습
        backbone = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        )
        # 분류 헤드를 이진 분류용으로 교체
        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 2)  # [real, fake]
        )
        self.backbone = backbone

    def forward(self, x):
        return self.backbone(x)

# 학습 흐름 (요약)
model = DeepfakeDetector().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

for epoch in range(10):
    model.train()
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x_batch), y_batch)
        loss.backward()
        optimizer.step()
    scheduler.step()

코드 domain-eval

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score
import torch

def evaluate_domain(model, loader, name):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y.numpy())
    acc = accuracy_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)
    print(f"{name:30s}  ACC={acc:.4f}  AUC={auc:.4f}")
    return acc, auc

# 5가지 도메인에서 평가
for loader, name in [
    (faceforensics_test, 'FaceForensics++ (학습 도메인)'),
    (faceshifter_test,   'FaceShifter (새 방법)'),
    (dfdc_test,          'DFDC (다른 데이터셋)'),
    (diffusion_test,     'Stable Diffusion 합성'),
    (kodf_test,          '한국어 KoDF')
]:
    evaluate_domain(model, loader, name)

코드 adversarial-attack

In [ ]:
import torch
import torch.nn.functional as F

def fgsm_attack(model, x, y, epsilon=0.01):
    """FGSM 공격: 사람 눈에 거의 보이지 않는 노이즈로 탐지 우회."""
    x = x.clone().detach().requires_grad_(True)
    logits = model(x)
    loss = F.cross_entropy(logits, y)
    loss.backward()

    x_adv = x + epsilon * x.grad.sign()
    x_adv = torch.clamp(x_adv, 0, 1).detach()
    return x_adv

# 합성 이미지 100장에 대한 공격 성공률
model.eval()
orig_correct = adv_correct = total = 0
for x, y in fake_loader_small:  # 모두 합성(label=1)
    x, y = x.to(device), y.to(device)

    orig_pred = model(x).argmax(dim=1)
    orig_correct += (orig_pred == y).sum().item()

    x_adv = fgsm_attack(model, x, y, epsilon=0.01)
    adv_pred = model(x_adv).argmax(dim=1)
    adv_correct += (adv_pred == y).sum().item()
    total += y.size(0)

print(f"공격 전 정확도: {orig_correct/total:.4f}")
print(f"FGSM ε=0.01 공격 후 정확도: {adv_correct/total:.4f}")